In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent 
sys.path.append(str(PROJECT_ROOT))

## Whisper Tokenizer

In [3]:
from src.data.tokenizers.whisper import WhisperTokenizer

tokenizer = WhisperTokenizer(language="vi")

sample_texts = ["Xin chào thế giới!"]
for text in sample_texts:
    ids = tokenizer.encode(text)
    print(f"Text: {text}")
    print(f"Token IDs: {ids}")
    decoded_text = tokenizer.batch_decode([ids])[0]
    print(f"Decoded Text: {decoded_text}")

WhisperTokenizer initialized
   Vocab size: 50258
   Language: vi
   Model: openai/whisper-tiny
Text: Xin chào thế giới!
Token IDs: [50258, 50278, 50359, 50363, 55, 259, 417, 20807, 27100, 1735, 12242, 0, 50257]
Decoded Text: Xin chào thế giới!


## Dataset

### Grid

In [4]:
import os
import glob
import json
import cv2

# Config
DATA_ROOT = "test_data" 
OUTPUT_MANIFEST = "manifest.jsonl"

# Find all .mpg video files 
search_pattern = os.path.join(DATA_ROOT, "s1_processed", "*.mpg")
video_files = glob.glob(search_pattern)

print(f"Found {len(video_files)} video files.")

entries = []
for video_path in video_files:
    
    # Find align file
    parent_dir = os.path.dirname(video_path) # test_data/s1_processed
    filename = os.path.basename(video_path)  # bbaf2n.mpg
    
    # Create path to align file: parent_dir -> align -> filename.align
    align_filename = filename.replace(".mpg", ".align")
    align_path = os.path.join(parent_dir, "align", align_filename)
    
    # Get Transcript
    transcript = ""
    if os.path.exists(align_path):
        try:
            with open(align_path, 'r') as f:
                words = [line.split()[2] for line in f if len(line.split()) >= 3 and line.split()[2] not in ['sil', 'sp']]
                transcript = " ".join(words)
        except Exception as e:
            print(f"Error reading align file {align_path}: {e}")
    else:
        # If still not found, print the path for debugging
        if len(entries) < 1: 
            print(f"Align file not found at: {align_path}")
    # Get Duration
    duration = 0.0
    try:
        cap = cv2.VideoCapture(video_path)
        if cap.isOpened():
            fps = cap.get(cv2.CAP_PROP_FPS)
            cnt = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            if fps > 0: duration = cnt / fps
        cap.release()
    except: pass

    # Create entry
    rel_path = os.path.relpath(video_path, DATA_ROOT)
    entries.append({
        "rel_path": rel_path,
        "text": transcript,
        "duration": round(duration, 2)
    })

# Write to manifest file
with open(OUTPUT_MANIFEST, 'w') as f:
    for e in entries:
        f.write(json.dumps(e) + "\n")

print(f" Created manifest: {OUTPUT_MANIFEST}")
# Print first few lines to verify transcripts are not empty
!head -n 2 {OUTPUT_MANIFEST}

Found 1000 video files.
 Created manifest: manifest.jsonl
{"rel_path": "s1_processed/prwq3s.mpg", "text": "place red with q three soon", "duration": 3.0}
{"rel_path": "s1_processed/pbib8p.mpg", "text": "place blue in b eight please", "duration": 3.0}


## Data Loader

In [5]:
!pip install openai-whisper


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
from src.data.loader import build_dataloader
from src.data.tokenizers.whisper import WhisperTokenizer

test_config = {
    "data_root": "test_data",
    "train_manifest": "manifest.jsonl",
    "val_manifest": "manifest.jsonl",
    "batch_size": 2,
    "num_workers": 0,
    "use_precomputed_features": False,
}


tokenizer = WhisperTokenizer(language="en")
train_loader = build_dataloader(test_config, tokenizer, mode='train')


print(f"Getting one batch from train_loader...")
batch = next(iter(train_loader))

audio = batch['audio']  # (B, T)
print(f"Audio batch shape: {audio.shape}")

visual = batch['visual']  # (B, C, H, W, T)
print(f"Visual batch shape: {visual.shape}")

targets = batch['target']  # (B, S)
print(f"Targets batch shape: {targets.shape}")

# Check labels
for i in range(targets.size(0)):
    # Only row token, remove padding -100 
    token_row = targets[i]
    token_ids = token_row[token_row != -100].tolist()

    # decode to text
    decoded_text = tokenizer.batch_decode([token_ids])[0]
    orginal_text = batch['text'][i]

    print(f"Sample {i}:")
    print(f"Raw tokens: {token_row.tolist()[:10]}")
    print(f"Original Text: {orginal_text}")
    print(f"Decoded Text: {decoded_text}")

WhisperTokenizer initialized
   Vocab size: 50258
   Language: en
   Model: openai/whisper-tiny
🔨 Building DataLoader [train]
   - Manifest: manifest.jsonl
   - Input Type: Raw Video (.mpg)
   - Batch Size: 2
Dataset GridDataset loaded: 1000 samples
Getting one batch from train_loader...


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Audio batch shape: torch.Size([2, 80, 3000])
Visual batch shape: torch.Size([2, 75, 3, 224, 224])
Targets batch shape: torch.Size([2, 11])
Sample 0:
Raw tokens: [50258, 50259, 50359, 50363, 8376, 3092, 294, 367, 4949, 2321]
Original Text: lay green in r nine soon
Decoded Text: lay green in r nine soon
Sample 1:
Raw tokens: [50258, 50259, 50359, 50363, 13496, 2418, 294, 275, 1732, 2321]
Original Text: bin white in m five soon
Decoded Text: bin white in m five soon


In [8]:
import os
import whisper
import cv2
import json

# 1. Cấu hình y hệt như lúc bạn test
DATA_ROOT = "test_data"
MANIFEST_PATH = "manifest.jsonl"

print(f"📍 Current working directory: {os.getcwd()}")

# 2. Đọc 1 dòng từ Manifest để lấy đường dẫn
try:
    with open(MANIFEST_PATH, 'r') as f:
        line = f.readline()
        data = json.loads(line)
        rel_path = data['rel_path'] # Ví dụ: s1_processed/bbaf2n.mpg
        
    full_path = os.path.join(DATA_ROOT, rel_path)
    print(f"🔍 Đang thử load file: {full_path}")
    
    # 3. Kiểm tra file có tồn tại không
    if not os.path.exists(full_path):
        print("❌ LỖI: File không tồn tại! Kiểm tra lại DATA_ROOT.")
    else:
        print("✅ File có tồn tại.")

        # 4. Thử load Audio (Thường chết ở đây nếu thiếu FFmpeg)
        print("   -> Đang thử load Audio bằng Whisper...")
        audio = whisper.load_audio(full_path)
        print(f"   -> ✅ Audio OK: {audio.shape}")

        # 5. Thử load Video
        print("   -> Đang thử load Video bằng OpenCV...")
        cap = cv2.VideoCapture(full_path)
        ret, frame = cap.read()
        if ret:
            print(f"   -> ✅ Video OK: {frame.shape}")
        else:
            print("   -> ❌ Lỗi đọc Video Frame!")
        cap.release()

except Exception as e:
    print("\n❌ LỖI CHI TIẾT:")
    print(e)

📍 Current working directory: /Volumes/Kingston_XS1000_Media/Audio-Visual-Speech-Recognition/notebooks
🔍 Đang thử load file: test_data/s1_processed/prwq3s.mpg
✅ File có tồn tại.
   -> Đang thử load Audio bằng Whisper...
   -> ✅ Audio OK: (47648,)
   -> Đang thử load Video bằng OpenCV...
   -> ✅ Video OK: (288, 360, 3)
